# 3 · ResNet18 fine-tuning
**Owner:** Amrit · **Phase 1:** GPU smoke test · **Phase 3:** training

On Colab: **Runtime → Change runtime type → T4 GPU**.

In [1]:
# ---- Setup: works on Colab and locally. Run this first. ----
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Shared Drive folder (add a shortcut to it in your "My Drive" so this path exists)
    DRIVE_DIR = Path("/content/drive/MyDrive/image-processing-group18")
    REPO_DIR = Path("/content/image-processing-group18")
    if not REPO_DIR.exists():
        # If the repo is private, use https://<YOUR_GITHUB_TOKEN>@github.com/... here
        subprocess.run(["git", "clone", "https://github.com/aylat7/image-processing-group18.git", str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # Copy the raw dataset from Drive to Colab's fast local disk (once per session)
    if not Path("data/raw/asl_raw").exists():
        os.makedirs("data/raw", exist_ok=True)
        subprocess.run(["unzip", "-q", str(DRIVE_DIR / "asl_raw.zip"), "-d", "data/raw"], check=True)
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
import common
print("Project root:", os.getcwd(), "| Colab:", IN_COLAB)

Project root: /home/roneet/Desktop/IMG467-project | Colab: False


## Phase 1 · GPU smoke test
Check that a pretrained ResNet18 loads, gets a 26-class head, and can train for a few steps on real images.

In [2]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (fine locally, use GPU on Colab)")

common.set_seed()
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, common.NUM_CLASSES)   # 512 -> 26
model = model.to(device)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

torch 2.15.0+cu132 | device: cpu | no GPU (fine locally, use GPU on Colab)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/roneet/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


Trainable params: 11189850


In [3]:
# A small batch of real images, run through the SAME preprocessing as the webcam app
paths, labels = [], []
for letter in common.CLASSES:
    for f in sorted((common.RAW_DIR / letter).glob("*.jpg"))[:2]:
        paths.append(f); labels.append(common.CLASS_TO_IDX[letter])
x = torch.from_numpy(np.stack([common.preprocess_cnn(common.load_image(p)) for p in paths])).to(device)
y = torch.tensor(labels).to(device)
print("batch:", tuple(x.shape), "labels:", tuple(y.shape))

opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()
model.train()
for step in range(10):
    t = time.time()
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()
    print(f"step {step}: loss {loss.item():.3f}  ({(time.time() - t) * 1000:.0f} ms)")
print("Loss should go down -> the training loop works.")

batch: (52, 3, 224, 224) labels: (52,)


step 0: loss 3.404  (2463 ms)


step 1: loss 2.284  (2407 ms)


step 2: loss 1.499  (2388 ms)


step 3: loss 0.974  (2390 ms)


step 4: loss 0.642  (2349 ms)


step 5: loss 0.431  (2367 ms)


step 6: loss 0.295  (2387 ms)


step 7: loss 0.207  (2661 ms)


step 8: loss 0.151  (3283 ms)


step 9: loss 0.115  (3891 ms)
Loss should go down -> the training loop works.


In [4]:
# Export / reload round-trip (this is how the webcam app will load it on CPU)
import tempfile, os
tmp = os.path.join(tempfile.gettempdir(), "smoke_test.pt")
torch.save(model.state_dict(), tmp)
m2 = resnet18(weights=None); m2.fc = nn.Linear(512, common.NUM_CLASSES)
m2.load_state_dict(torch.load(tmp, map_location="cpu"))
print("Reloaded on CPU OK, file size:", round(os.path.getsize(tmp) / 1e6, 1), "MB")

Reloaded on CPU OK, file size: 44.8 MB


---
## Phase 3 · Training

Fine-tunes the ImageNet-pretrained ResNet18 with a 26-letter head (as in the proposal) and exports `resnet18.pt` for the webcam app.

**Needs first:** Ryan's `data/splits/train.csv`, `val.csv`, `test.csv` (Phase 2), committed to the repo. Then `Runtime > Run all` on a T4 GPU (~20 to 30 min).

| Output | Used by |
|---|---|
| `models/resnet18.pt` | `webcam_app.py` (Ayla), CPU timing (Roneet + Ayla) |
| `results/resnet18/predictions_test.csv`, `metrics.json` | `4_compare.ipynb` (Roneet) |
| `results/resnet18/*.png` (curves, confusion matrix, examples, Grad-CAM) | Midterm and final report |
| `checkpoints/resnet18/*.pt` | Resuming after a Colab disconnect |

On Colab these are written to the shared Drive folder (`image-processing-group18/`); locally to `models/` and `results/`.

**Project rules this section enforces**
1. All preprocessing goes through `common.preprocess_cnn`, the same function the webcam app calls.
2. Images come **only** from the shared split CSVs, so HOG + SVM and ResNet18 see identical data.
3. It stops with an error if any split row points into `webcam_test`.
4. A checkpoint is saved **every epoch** and training resumes automatically after a disconnect.

In [ ]:
import json, csv, math, random, shutil, time
import numpy as np, cv2, torch, torch.nn as nn, torchvision
import matplotlib.pyplot as plt
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

for _v in ("model", "m2", "opt", "x", "y"):          # free the Phase 1 smoke-test objects
    globals().pop(_v, None)
if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"                     # mixed precision on GPU only
torch.backends.cudnn.benchmark = True

# Outputs: shared Drive on Colab (survives the session), repo folders locally (models/ is gitignored)
OUT_ROOT    = DRIVE_DIR if IN_COLAB else common.PROJECT_ROOT
MODELS_DIR  = DRIVE_DIR / "models" if IN_COLAB else common.MODELS_DIR
CKPT_DIR    = (DRIVE_DIR / "checkpoints" if IN_COLAB else common.MODELS_DIR / "checkpoints") / "resnet18"
RESULTS_DIR = OUT_ROOT / "results" / "resnet18"
for d in (MODELS_DIR, CKPT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED, CLASSES, NUM_CLASSES, CLASS_TO_IDX = common.SEED, common.CLASSES, common.NUM_CLASSES, common.CLASS_TO_IDX
preprocess_cnn = common.preprocess_cnn
common.set_seed()
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (use a T4 GPU on Colab)")
print("Checkpoints:", CKPT_DIR, "| results:", RESULTS_DIR)

### Hyperparameters

Everything tunable lives here so the report can cite one place. Choices are tuned on the **validation split only**.

| Setting | Value | Why |
|---|---|---|
| Fine-tuning | 2 stages: head only, then full network | Training the new 26-class layer first stops large random gradients from damaging the pretrained features |
| Optimizer | AdamW, weight decay 1e-4 | Stable default for fine-tuning; decoupled weight decay |
| Learning rate | Stage 1: 1e-3 · Stage 2: backbone 1e-4, head 5e-4 | Pretrained layers need smaller updates than the new layer |
| Schedule | 1 epoch linear warm-up, then cosine decay | Smooth convergence, no manual step tuning |
| Loss | Cross-entropy with label smoothing 0.1 | Softer targets reduce over-confidence, which helps on unseen webcam images |
| Model selection | Best **validation macro-F1**, early stopping (patience 4) | Macro-F1 is one of our two headline metrics and weights every letter equally |

In [ ]:
HP = dict(
    batch_size     = 64,
    epochs_head    = 3,      # stage 1: only the new fc layer trains
    epochs_full    = 15,     # stage 2: whole network (early stopping may end it sooner)
    lr_head_stage1 = 1e-3,
    lr_backbone    = 1e-4,
    lr_head_stage2 = 5e-4,
    weight_decay   = 1e-4,
    label_smoothing= 0.1,
    warmup_epochs  = 1,
    patience       = 4,
    num_workers    = 2,
)
print(HP)

# Check the common.py output the model expects (fails loudly if someone changes it)
_probe = preprocess_cnn(np.full((300, 260, 3), 128, np.uint8))
assert _probe.dtype == np.float32 and _probe.shape == (3, common.CNN_IMG_SIZE, common.CNN_IMG_SIZE), _probe.shape

def to_display(chw):   # only for figures: undo the ImageNet normalization from common.py
    return np.clip(chw.transpose(1, 2, 0) * common.IMAGENET_STD + common.IMAGENET_MEAN, 0, 1)

### Load the shared splits and check them

Reads `data/splits/{train,val,test}.csv` (columns `path,label`, paths relative to `data/`). Webcam training images reach this notebook **only through these files**, so both models always train on the same data.

Checks: every file exists, all 26 letters appear in every split, no image appears in two splits, and nothing points into `webcam_test`. Files missing on Colab's local disk (e.g. `webcam_train/`) are copied once from the shared Drive `data/` folder.

In [ ]:
DATA_DIR = common.DATA_DIR

def read_split(name):
    rows = []
    with open(common.SPLITS_DIR / f"{name}.csv", newline="") as f:
        for r in csv.DictReader(f):
            lab = r["label"].strip()
            y = CLASS_TO_IDX[lab.upper()] if lab.isalpha() else int(lab)
            rows.append((r["path"].strip().replace("\\", "/"), y))
    return rows

missing_csv = [s for s in ("train", "val", "test") if not (common.SPLITS_DIR / f"{s}.csv").exists()]
assert not missing_csv, f"Waiting on Phase 2: {missing_csv} not in data/splits/ yet (git pull after Ryan commits them)"
splits = {s: read_split(s) for s in ("train", "val", "test")}
all_paths = [p for rows in splits.values() for p, _ in rows]

assert not any("webcam_test" in p for p in all_paths), "RULE 3 VIOLATION: a split file references webcam_test"
for a in splits:
    for b in splits:
        if a < b:
            overlap = {p for p, _ in splits[a]} & {p for p, _ in splits[b]}
            assert not overlap, f"{len(overlap)} images appear in both {a} and {b}"

missing = [p for p in all_paths if not (DATA_DIR / p).exists()]
if missing and IN_COLAB:   # e.g. webcam_train/ lives on Drive, not in asl_raw.zip
    for p in missing:
        src = DRIVE_DIR / "data" / p
        if src.exists():
            (DATA_DIR / p).parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, DATA_DIR / p)
    missing = [p for p in all_paths if not (DATA_DIR / p).exists()]
assert not missing, f"{len(missing)} files listed in splits are missing, e.g. {missing[:3]}"

print(f"{'split':6} {'images':>7}  min/max per class")
for s, rows in splits.items():
    counts = np.bincount([y for _, y in rows], minlength=NUM_CLASSES)
    assert (counts > 0).all(), f"{s} is missing letters: {[CLASSES[i] for i in np.where(counts == 0)[0]]}"
    print(f"{s:6} {len(rows):7d}  {counts.min()}/{counts.max()}")

### Dataset and training-only augmentation

Augmentation is applied to the raw BGR image **before** `preprocess_cnn`, so the final resize and normalization stay identical to the webcam app. Each transform targets a condition from our trial protocol:

| Augmentation | Range | Targets |
|---|---|---|
| Brightness / contrast | ×0.6 to 1.4, ±40 | Normal vs **dim lighting** |
| Gamma | 0.7 to 1.5 | Non-linear exposure of cheap webcams |
| Hue / saturation | ±6 hue, ×0.7 to 1.3 sat | Skin tone and white-balance differences between participants |
| Rotation, scale, shift | ±12°, ×0.9 to 1.1, ±8% | Hand not perfectly centred in the on-screen box |
| Blur / sensor noise | p = 0.2 each | Webcam softness and low-light grain |

**No horizontal flip.** Mirroring turns a right-hand sign into a left-hand one and changes how letters such as J and Z appear, so it would add label noise instead of variety.

Randomness is seeded by `(seed, epoch, index)`, so augmentations are reproducible and independent of the number of workers.

In [ ]:
def augment_bgr(img, rng):
    h, w = img.shape[:2]
    # geometry: rotation + scale + translation in one affine warp
    ang = rng.uniform(-12, 12); sc = rng.uniform(0.9, 1.1)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    M[:, 2] += rng.uniform(-0.08, 0.08, 2) * (w, h)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    # colour: hue / saturation
    if rng.random() < 0.8:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 0] = (hsv[..., 0] + rng.uniform(-6, 6)) % 180
        hsv[..., 1] = np.clip(hsv[..., 1] * rng.uniform(0.7, 1.3), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    # lighting: brightness / contrast, then gamma
    img = cv2.convertScaleAbs(img, alpha=rng.uniform(0.6, 1.4), beta=rng.uniform(-40, 40))
    if rng.random() < 0.5:
        lut = (np.linspace(0, 1, 256) ** rng.uniform(0.7, 1.5) * 255).astype(np.uint8)
        img = cv2.LUT(img, lut)
    # webcam artefacts
    if rng.random() < 0.2:
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.5, 1.5))
    if rng.random() < 0.2:
        img = np.clip(img + rng.normal(0, rng.uniform(3, 10), img.shape), 0, 255).astype(np.uint8)
    return img


class ASLDataset(torch.utils.data.Dataset):
    def __init__(self, rows, train):
        self.rows, self.train, self.epoch = rows, train, 0
    def set_epoch(self, e): self.epoch = e
    def __len__(self): return len(self.rows)
    def load_bgr(self, i):
        img = common.load_image(DATA_DIR / self.rows[i][0])
        if self.train:
            img = augment_bgr(img, np.random.default_rng((SEED, self.epoch, i)))
        return img
    def __getitem__(self, i):
        return torch.from_numpy(preprocess_cnn(self.load_bgr(i))), self.rows[i][1]


def worker_init(_):
    cv2.setNumThreads(0)   # avoid CPU oversubscription inside DataLoader workers

def make_loader(ds, shuffle):
    g = torch.Generator(); g.manual_seed(SEED)
    return torch.utils.data.DataLoader(
        ds, batch_size=HP["batch_size"], shuffle=shuffle, num_workers=HP["num_workers"],
        pin_memory=DEVICE.type == "cuda", worker_init_fn=worker_init, generator=g,
        persistent_workers=False, drop_last=False)  # non-persistent so set_epoch reaches workers

ds = {s: ASLDataset(rows, train=(s == "train")) for s, rows in splits.items()}
loaders = {s: make_loader(d, shuffle=(s == "train")) for s, d in ds.items()}
print({s: len(d) for s, d in ds.items()})

In [ ]:
# Sanity check: the same training images with and without augmentation, shown exactly as the model receives them
idx = np.random.default_rng(0).choice(len(ds["train"]), 6, replace=False)
clean = ASLDataset(splits["train"], train=False)
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for j, i in enumerate(idx):
    ax[0, j].imshow(to_display(clean[i][0].numpy())); ax[0, j].set_title(f"{CLASSES[clean[i][1]]} original")
    ax[1, j].imshow(to_display(ds["train"][i][0].numpy())); ax[1, j].set_title("augmented")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "augmentation_examples.png", dpi=150); plt.show()

### Model

ResNet18 with ImageNet weights; the 1000-class `fc` layer is replaced by `Linear(512, 26)`, exactly as in the proposal. In stage 1 the backbone is frozen **including its BatchNorm statistics**, so the small new head cannot shift the pretrained normalisation.

In [ ]:
def build_model(pretrained=True):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def set_stage(model, stage):
    for name, p in model.named_parameters():
        p.requires_grad = (stage == "full") or name.startswith("fc.")

def train_mode(model, stage):
    model.train()
    if stage == "head":   # keep pretrained BatchNorm running stats frozen while only the head trains
        for mod in model.modules():
            if isinstance(mod, nn.BatchNorm2d): mod.eval()

model = build_model().to(DEVICE).to(memory_format=torch.channels_last)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])

### Training engine with per-epoch Drive checkpoints

* `stage{1,2}_last.pt` holds model, optimizer, scheduler, AMP scaler and history, written **every epoch** (atomic write, so a disconnect mid-save cannot corrupt it).
* `best.pt` holds the weights with the highest validation macro-F1 so far.
* Re-running the notebook after a disconnect resumes from the last finished epoch; a finished stage is skipped.

In [ ]:
def atomic_save(obj, path):
    tmp = Path(str(path) + ".tmp"); torch.save(obj, tmp); os.replace(tmp, path)

def make_optimizer(model, stage):
    if stage == "head":
        groups = [{"params": model.fc.parameters(), "lr": HP["lr_head_stage1"]}]
    else:
        groups = [{"params": [p for n, p in model.named_parameters() if not n.startswith("fc.")], "lr": HP["lr_backbone"]},
                  {"params": model.fc.parameters(), "lr": HP["lr_head_stage2"]}]
    return torch.optim.AdamW(groups, weight_decay=HP["weight_decay"])

def make_scheduler(opt, epochs, steps_per_epoch):
    total, warm = epochs * steps_per_epoch, HP["warmup_epochs"] * steps_per_epoch
    def f(step):
        if step < warm: return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); losses, probs, ys = [], [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        losses.append(criterion(logits.float(), y).item() * len(y))
        probs.append(torch.softmax(logits.float(), 1).cpu()); ys.append(y.cpu())
    probs, ys = torch.cat(probs).numpy(), torch.cat(ys).numpy(); pred = probs.argmax(1)
    return dict(loss=sum(losses) / len(ys), acc=float(accuracy_score(ys, pred)),
                macro_f1=float(f1_score(ys, pred, average="macro")), probs=probs, y=ys, pred=pred)

def run_stage(model, stage, epochs, state):
    last_path = CKPT_DIR / f"stage{1 if stage == 'head' else 2}_last.pt"
    set_stage(model, stage)
    opt = make_optimizer(model, stage)
    sched = make_scheduler(opt, epochs, len(loaders["train"]))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    start, bad = 0, 0
    if last_path.exists():
        ck = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        state.update(ck["state"]); start, bad = ck["epoch"] + 1, ck["bad"]
        if ck.get("done"):
            print(f"[{stage}] already finished, skipping"); return
        print(f"[{stage}] resuming at epoch {start + 1}/{epochs}")

    for ep in range(start, epochs):
        ds["train"].set_epoch(ep + (100 if stage == "full" else 0))
        train_mode(model, stage); t0 = time.time(); tl, tc, n = 0.0, 0, 0
        for x, y in loaders["train"]:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x); loss = criterion(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); tc += (logits.argmax(1) == y).sum().item(); n += len(y)
        v = evaluate(model, loaders["val"])
        rec = dict(stage=stage, epoch=ep + 1, train_loss=tl / n, train_acc=tc / n,
                   val_loss=v["loss"], val_acc=v["acc"], val_macro_f1=v["macro_f1"], secs=time.time() - t0)
        state["history"].append(rec)
        improved = v["macro_f1"] > state["best_f1"]
        if improved:
            state["best_f1"] = v["macro_f1"]; bad = 0
            atomic_save({"model": model.state_dict(), "epoch": ep + 1, "stage": stage, "val_macro_f1": v["macro_f1"]},
                        CKPT_DIR / "best.pt")
        else:
            bad += 1
        stop = stage == "full" and bad >= HP["patience"]
        atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                     "scaler": scaler.state_dict(), "epoch": ep, "bad": bad, "state": state,
                     "done": stop or ep == epochs - 1}, last_path)
        print(f"[{stage}] ep {ep + 1:2d}/{epochs} | train loss {rec['train_loss']:.3f} acc {rec['train_acc']:.3f} | "
              f"val loss {rec['val_loss']:.3f} acc {rec['val_acc']:.3f} F1 {rec['val_macro_f1']:.3f}"
              f"{'  *best*' if improved else ''} | {rec['secs']:.0f}s")
        if stop:
            print(f"[{stage}] early stopping: no val macro-F1 gain for {HP['patience']} epochs"); break

### Train

In [ ]:
state = {"history": [], "best_f1": -1.0}
run_stage(model, "head", HP["epochs_head"], state)

# Stage 2 starts from the best weights so far (only if stage 2 has not started yet)
if not (CKPT_DIR / "stage2_last.pt").exists():
    model.load_state_dict(torch.load(CKPT_DIR / "best.pt", map_location=DEVICE, weights_only=False)["model"])
run_stage(model, "full", HP["epochs_full"], state)

with open(RESULTS_DIR / "history.json", "w") as f: json.dump(state["history"], f, indent=1)
print(f"Best validation macro-F1: {state['best_f1']:.4f}")

In [ ]:
H = state["history"]; ep = np.arange(1, len(H) + 1)
n_head = sum(h["stage"] == "head" for h in H)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (key, title) in zip(ax, [("loss", "Cross-entropy loss"), ("acc", "Accuracy"), ("macro_f1", "Macro-F1")]):
    if key != "macro_f1": a.plot(ep, [h[f"train_{key}"] for h in H], "o-", label="train (augmented)")
    a.plot(ep, [h[f"val_{key}"] for h in H], "s-", label="validation")
    a.axvline(n_head + 0.5, color="gray", ls="--", lw=1); a.text(n_head + 0.6, a.get_ylim()[1], " full fine-tune", va="top", fontsize=8)
    a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150); plt.show()

### Test-set evaluation (dataset test split)

Loads `best.pt` (chosen on validation only) and evaluates **once** on the held-out test split with the metrics from our proposal: accuracy, macro-F1 and the confusion matrix. Webcam trial results are computed in `4_compare.ipynb`.

In [ ]:
best = torch.load(CKPT_DIR / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model"])
val_res, test_res = evaluate(model, loaders["val"]), evaluate(model, loaders["test"])
y, pred, probs = test_res["y"], test_res["pred"], test_res["probs"]

print(f"Best checkpoint: stage '{best['stage']}', epoch {best['epoch']}")
print(f"Validation  acc {val_res['acc']:.4f} | macro-F1 {val_res['macro_f1']:.4f}")
print(f"Test        acc {test_res['acc']:.4f} | macro-F1 {test_res['macro_f1']:.4f}\n")
report = classification_report(y, pred, labels=range(NUM_CLASSES), target_names=CLASSES, digits=3, output_dict=True, zero_division=0)
print(classification_report(y, pred, labels=range(NUM_CLASSES), target_names=CLASSES, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y, pred, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES), CLASSES); ax.set_yticks(range(NUM_CLASSES), CLASSES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cmn[i, j] >= 0.01:
            ax.text(j, i, f"{100 * cmn[i, j]:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted letter"); ax.set_ylabel("True letter")
ax.set_title(f"ResNet18, dataset test split (row %)  |  acc {test_res['acc']:.3f}, macro-F1 {test_res['macro_f1']:.3f}")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.savefig(RESULTS_DIR / "confusion_matrix_test.png", dpi=150); plt.show()

# Most frequent confusions
off = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
top_conf = sorted(off, reverse=True)[:10]
print("Top confusions (true -> predicted: count, % of true letter)")
for c, t, p in top_conf:
    print(f"  {t} -> {p}: {c:3d}  ({100 * c / cm[CLASS_TO_IDX[t]].sum():.1f}%)")

# Hypothesis from the proposal: held J is confused with I, held Z with D
jz = {}
for letter, partner in (("J", "I"), ("Z", "D")):
    i, k = CLASS_TO_IDX[letter], CLASS_TO_IDX[partner]
    row = cm[i].copy(); row[i] = 0
    jz[letter] = dict(recall=float(cmn[i, i]), to_partner=float(cmn[i, k]), from_partner=float(cmn[k, i]),
                      top_wrong=CLASSES[int(row.argmax())] if row.max() > 0 else "none")
    print(f"{letter}: recall {100 * cmn[i, i]:.1f}% | {letter}->{partner} {100 * cmn[i, k]:.1f}% | "
          f"{partner}->{letter} {100 * cmn[k, i]:.1f}% | most common wrong prediction: {jz[letter]['top_wrong']}")

### Qualitative results: where it works and where it fails

The midterm report asks for sample outputs of both. The top row shows confident correct predictions; the bottom row shows the most confident **mistakes**, which are the most informative for error analysis.

In [ ]:
conf = probs.max(1); right, wrong = np.where(pred == y)[0], np.where(pred != y)[0]
rng = np.random.default_rng(SEED)
show_right = rng.choice(right, min(6, len(right)), replace=False) if len(right) else []
show_wrong = wrong[np.argsort(-conf[wrong])][:6]
fig, ax = plt.subplots(2, 6, figsize=(15, 5.6))
for row, sel, col in ((0, show_right, "green"), (1, show_wrong, "red")):
    for j in range(6):
        a = ax[row, j]; a.axis("off")
        if j < len(sel):
            i = sel[j]; a.imshow(to_display(ds["test"][i][0].numpy()))
            a.set_title(f"true {CLASSES[y[i]]} | pred {CLASSES[pred[i]]} ({conf[i]:.2f})", color=col, fontsize=9)
ax[0, 0].text(-0.15, 0.5, "Correct", transform=ax[0, 0].transAxes, rotation=90, va="center", fontsize=12, color="green")
ax[1, 0].text(-0.15, 0.5, "Errors", transform=ax[1, 0].transAxes, rotation=90, va="center", fontsize=12, color="red")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "examples_correct_vs_errors.png", dpi=150); plt.show()
print(f"{len(wrong)} errors out of {len(y)} test images")

#### Grad-CAM: what is the network looking at?

Grad-CAM (Selvaraju et al., 2017), written directly in PyTorch with hooks on the last residual block, highlights the image regions that drove a prediction. If the heat sits on the **fingers**, the model learned the handshape; if it sits on the **background**, it learned a shortcut that will break on cluttered webcam scenes. This supports our hypothesis that ResNet18 handles clutter better than HOG + SVM.

In [ ]:
def grad_cam(model, x_chw, target):
    store = {}
    def fwd_hook(m, i, o):
        store["act"] = o; o.register_hook(lambda g: store.__setitem__("grad", g))
    h = model.layer4.register_forward_hook(fwd_hook)
    model.eval(); model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        x = torch.from_numpy(x_chw)[None].to(DEVICE).requires_grad_(True)
        model(x)[0, target].backward()
    h.remove()
    a, g = store["act"][0].float(), store["grad"][0].float()
    cam = torch.relu((g.mean((1, 2))[:, None, None] * a).sum(0)).detach().cpu().numpy()
    cam = cv2.resize(cam / (cam.max() + 1e-8), (x_chw.shape[2], x_chw.shape[1]))
    return cam

def overlay(x_chw, cam):
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1] / 255.0
    return np.clip(0.55 * to_display(x_chw) + 0.45 * heat, 0, 1)

sel = list(show_right[:3]) + list(show_wrong[:3])
fig, ax = plt.subplots(2, len(sel), figsize=(2.6 * len(sel), 5.4))
for j, i in enumerate(sel):
    x = ds["test"][i][0].numpy()
    ax[0, j].imshow(to_display(x)); ax[0, j].set_title(f"true {CLASSES[y[i]]} | pred {CLASSES[pred[i]]}",
                                                       color="green" if pred[i] == y[i] else "red", fontsize=9)
    ax[1, j].imshow(overlay(x, grad_cam(model, x, int(pred[i])))); ax[1, j].set_title("Grad-CAM (predicted letter)", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "gradcam_examples.png", dpi=150); plt.show()

### Save predictions and metrics for `4_compare.ipynb`

`predictions_test.csv` uses the same row order and paths as `splits/test.csv`, so Roneet can join it with the HOG + SVM predictions image by image.

In [ ]:
with open(RESULTS_DIR / "predictions_test.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["path", "true", "pred", "confidence", "top3"])
    for (p, _), t, pr, pb in zip(splits["test"], y, pred, probs):
        w.writerow([p, CLASSES[t], CLASSES[pr], f"{pb.max():.4f}", " ".join(CLASSES[k] for k in np.argsort(-pb)[:3])])

metrics = dict(
    model="ResNet18 (ImageNet-pretrained, fine-tuned)", best_stage=best["stage"], best_epoch=best["epoch"],
    val=dict(acc=val_res["acc"], macro_f1=val_res["macro_f1"]),
    test=dict(acc=test_res["acc"], macro_f1=test_res["macro_f1"], n=int(len(y))),
    per_class_f1={c: report[c]["f1-score"] for c in CLASSES},
    top_confusions=[dict(true=t, pred=p, count=int(c)) for c, t, p in top_conf],
    jz_analysis=jz, confusion_matrix=cm.tolist(), hparams=HP, seed=SEED,
    env=dict(torch=str(torch.__version__), torchvision=str(torchvision.__version__),
             gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"),
)
with open(RESULTS_DIR / "metrics.json", "w") as f: json.dump(metrics, f, indent=1)
print("Saved:", sorted(p.name for p in RESULTS_DIR.iterdir()))

### Export `resnet18.pt` for the webcam app

The file stores plain weights plus the metadata the app needs. Loading it needs only PyTorch and torchvision, on CPU:

```python
import torch, torch.nn as nn, common
from torchvision.models import resnet18
ck = torch.load("models/resnet18.pt", map_location="cpu")
model = resnet18(weights=None); model.fc = nn.Linear(512, len(ck["classes"]))
model.load_state_dict(ck["state_dict"]); model.eval()
x = torch.from_numpy(common.preprocess_cnn(crop_bgr))[None]
with torch.no_grad(): probs = torch.softmax(model(x), 1)[0]
letter, conf = ck["classes"][int(probs.argmax())], float(probs.max())
```

The cell below reloads the exported file exactly this way, on CPU, and checks that it reproduces the GPU predictions.

In [ ]:
export = dict(arch="resnet18", classes=CLASSES, input_size=224, preprocess="common.preprocess_cnn",
              state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
              val_macro_f1=float(val_res["macro_f1"]), test_acc=float(test_res["acc"]),
              test_macro_f1=float(test_res["macro_f1"]), trained_epoch=int(best["epoch"]),
              torch_version=str(torch.__version__))   # plain types only, so torch.load works with its safe default
OUT = MODELS_DIR / "resnet18.pt"; atomic_save(export, OUT)
print(f"Exported {OUT} ({OUT.stat().st_size / 1e6:.1f} MB)")

# Round-trip check on CPU, the way webcam_app.py will load it
ck = torch.load(OUT, map_location="cpu")
cpu_model = resnet18(weights=None); cpu_model.fc = nn.Linear(512, len(ck["classes"]))
cpu_model.load_state_dict(ck["state_dict"]); cpu_model.eval()
check = np.arange(min(64, len(ds["test"])))
xb = torch.stack([ds["test"][i][0] for i in check])
with torch.no_grad(): cpu_pred = cpu_model(xb).argmax(1).numpy()
agree = (cpu_pred == pred[check]).mean()
print(f"CPU reload agrees with GPU predictions on {100 * agree:.1f}% of {len(check)} images")
assert agree >= 0.98, "Exported model does not reproduce training-time predictions"

### Summary for the report

In [ ]:
H = state["history"]
print("Copy into the mid-progress report (Preliminary Results):\n")
print("| Model | Split | Accuracy | Macro-F1 |")
print("|---|---|---|---|")
print(f"| ResNet18 | Validation | {val_res['acc']:.3f} | {val_res['macro_f1']:.3f} |")
print(f"| ResNet18 | Test (dataset) | {test_res['acc']:.3f} | {test_res['macro_f1']:.3f} |")
f1s = sorted(((report[c]["f1-score"], c) for c in CLASSES))
print(f"\nEpochs run: {len(H)} ({sum(h['stage'] == 'head' for h in H)} head-only + {sum(h['stage'] == 'full' for h in H)} full), "
      f"best at {best['stage']} epoch {best['epoch']}, avg {np.mean([h['secs'] for h in H]):.0f}s/epoch on {metrics['env']['gpu']}")
print("Weakest letters (F1):", ", ".join(f"{c} {v:.2f}" for v, c in f1s[:4]))
print("Strongest letters (F1):", ", ".join(f"{c} {v:.2f}" for v, c in f1s[-4:]))
print("Top confusions:", ", ".join(f"{t}->{p} ({c})" for c, t, p in top_conf[:5]))
print(f"J: recall {jz['J']['recall']:.2f}, J->I {jz['J']['to_partner']:.2f} | Z: recall {jz['Z']['recall']:.2f}, Z->D {jz['Z']['to_partner']:.2f}")

## Next steps (Phase 4)

* **Roneet:** `4_compare.ipynb` reads `results/resnet18/predictions_test.csv` and `metrics.json`, and runs `resnet18.pt` on `webcam_test` per lighting × background condition.
* **Ryan + Amrit:** error analysis using the confusion matrix, the error grid and Grad-CAM (does the model fail on clutter, dim light, or similar handshapes?).
* **Ayla:** load `models/resnet18.pt` in `webcam_app.py` with the snippet in the export section.